In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import thin, support_mask, generate_single_cell, stamp_cell
# from render import localization_function, render, texture, build_comps, N_POOLS, POOL_NAMES, field
from render import localization_function, render_marker, pool_image, build_comps, render_cell, N_POOLS, POOL_NAMES, field
from tape import Tape
from parameter import P
from plot import plot_polar_phi, plot_tau
    

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
# Draw Tape. For the later optimization, we want to draw the random numbers once, and then use them for all parameter combinations. This is to avoid the random numbers changing when we change parameters.
SEED=None
SIZE=201
K=20
TILE = 256
N_CAND = 1500

tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile = TILE, n_cand = N_CAND, Pool = N_POOLS)


## 1.1) Cell shape gen

In [ ]:
# implement cells as star shaped polygons.
cell = generate_single_cell(tape)

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
axes[0].imshow(cell['rho'])
axes[0].set_title('Rho')
axes[1].imshow(cell['phi'])
axes[1].set_title('Phi')
axes[2].imshow(cell['tau'])
axes[2].set_title('Tau')
axes[3].imshow(cell['cell'])
axes[3].set_title('Cell')
axes[4].imshow(cell['nuc'])
axes[4].set_title('Nuc')
plt.show()


In [ ]:
def plot_interactive_cell(size, radius, nuc_frac, rough, elong, angle_deg, beta):
    # Generate the masks
    cell = generate_single_cell(tape,
        size=size, radius=radius, nuc_frac=nuc_frac, 
        rough=rough, elong=elong, 
        angle_deg=angle_deg, beta=beta
    )
    
    # Create an RGB image background (black)
    img = np.zeros((size, size, 3))
    
    # Assign colors using the masks
    # Cytoplasm (light green)
    img[cell['cell']] = [0.2, 0.8, 0.3]
    # Nucleus (light blue) overwrites the cytoplasm where it exists
    img[cell['nuc']] = [0.3, 0.5, 0.9]
    
    # Plotting
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Synthetic Cell Generation')
    plt.show()
# Set up the interactive sliders with reasonable bounds based on your defaults
interact(plot_interactive_cell,
         size=widgets.IntSlider(min=100, max=500, step=10, value=201, description='Size'),
         radius=widgets.FloatSlider(min=10.0, max=150.0, step=1.0, value=32.0, description='Radius'),
         nuc_frac=widgets.FloatSlider(min=0.1, max=0.9, step=0.05, value=0.3, description='Nuc Frac'),
         rough=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=0.9, description='Roughness'),
         elong=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.6, description='Elongation'),
         angle_deg=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=30.0, description='Angle (deg)'),
         beta=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.9, description='Beta'));

## 1.2) Cell Marker expression

### 1.2.2) Single Marker

In [ ]:
cell = generate_single_cell(tape)
plot_tau(cell['tau'], ~cell["cell"])

In [ ]:
def plot_combined(cell, tape, amp, polarity, pol_dir, **slider_kw):
    cell, phi, tau = cell['cell'], cell['phi'], cell['tau']
    comps = build_comps(**slider_kw)
    img = render_marker(comps, cell, tau, phi, tape, polarity, pol_dir, amp)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8.6), height_ratios=[1.15, 1.6])
    cols = ["#3b5bdb", "#e8590c", "#2f9e44"]
    x = np.linspace(-1, 1, 301)
    tot = sum(c["w"] for c in comps) or 1.0
    for j, c in enumerate(comps):
        axes[0, 0].plot(x, localization_function(x, c["mu"], c["width"], c["sharp"]),
                        lw=2.2, color=cols[j], alpha=max(c["w"] / tot, .12),
                        label=f"{POOL_NAMES[j]}  w={c['w']/tot:.2f}")
    axes[0, 0].axvline(0, color="0.6", lw=.8); axes[0, 0].axvline(1, color="0.6", lw=.8)
    axes[0, 0].set_xlabel(r"$\tau$"); axes[0, 0].set_ylim(bottom=0)
    axes[0, 0].grid(ls="--", alpha=.4); axes[0, 0].legend(fontsize=8)
    axes[0, 0].set_title("where each pool sits", fontsize=10)

    hi = np.percentile(img[cell], 99.5)
    im = axes[0, 1].imshow(np.where(cell, img, np.nan), vmin=0, vmax=hi, cmap="viridis")
    axes[0, 1].set_title("final   mean=%.2f   peak/mean=%.1f"
                         % (img[cell].mean(), img[cell].max() / img[cell].mean()), fontsize=10)
    fig.colorbar(im, ax=axes[0, 1], fraction=.046, pad=.04)

    axes[0, 2].barh(range(N_POOLS), [c["w"] / tot for c in comps], color=cols)
    axes[0, 2].set_yticks(range(N_POOLS)); axes[0, 2].set_yticklabels(POOL_NAMES, fontsize=9)
    axes[0, 2].set_xlim(0, 1); axes[0, 2].set_title("normalised weights", fontsize=10)
    axes[0, 2].invert_yaxis()

    for j, c in enumerate(comps):
        p = pool_image(c, cell, tau, tape["noise"][j])
        axes[1, j].imshow(np.where(cell, p, np.nan), cmap="viridis",
                          vmin=0, vmax=np.percentile(p[cell], 99.5))
        axes[1, j].set_title("%s  $\\mu$=%.2f  w=%.2f" % (POOL_NAMES[j], c["mu"], c["w"] / tot), fontsize=9)

    for A in [axes[0, 1]] + list(axes[1]):
        A.axis("off")
    fig.tight_layout()
    return fig


# from single_cell import single_cell
cell = generate_single_cell(size = 201, Tape = tape)

# interactive figure
F = lambda lo, hi, st, v, d: W.FloatSlider(min=lo, max=hi, step=st, value=v,
                                           description=d, continuous_update=False)
ctl = dict(
    amp=F(.1, 3, .1, 1.0, "amp"),
    polarity=F(-2, 2, .1, .4, "polarity"),
    pol_dir=F(-np.pi, np.pi, .1, .7, "pol dir"),

    w0=F(0, 1, .05, .50, "weight"),      s0=F(0, 2, .05, .4, "strength"),
    mu0=F(-1, 1.5, .05, 1.00, "mu"),     width0=F(.05, 1.5, .05, .25, "width"),
    sharp0=F(.5, 10, .5, 2.5, "sharp"),  scale0=F(.5, 12, .5, 4.0, "blob scale"),

    w1=F(0, 1, .05, .35, "weight"),      s1=F(0, 2, .05, 1.0, "strength"),
    mu1=F(-1, 1.5, .05, .45, "mu"),      width1=F(.05, 1.5, .05, .50, "width"),
    sharp1=F(.5, 10, .5, 2.0, "sharp"),  lam1=F(2, 25, .5, 8.0, "stripe lam"),
    len1=F(3, 40, 1., 25., "stripe len"), ang1=F(0, 180, 5., 30., "stripe angle"),

    w2=F(0, 1, .05, .15, "weight"),      s2=F(0, 2, .05, 1.1, "strength"),
    mu2=F(-1, 1.5, .05, -.50, "mu"),     width2=F(.05, 1.5, .05, .50, "width"),
    sharp2=F(.5, 10, .5, 3.0, "sharp"),  scale2=F(.5, 5, .1, 1.3, "fine scale"),
)

out = W.Output()
def cb(**kw):
    with out:
        out.clear_output(wait=True)
        fig = plot_combined(cell, tape, **kw)
        display(fig); plt.close(fig)

tabs = W.Tab(children=[
    W.VBox([ctl[k] for k in ("amp", "polarity", "pol_dir")]),
    W.VBox([ctl[k] for k in ("w0", "s0", "mu0", "width0", "sharp0", "scale0")]),
    W.VBox([ctl[k] for k in ("w1", "s1", "mu1", "width1", "sharp1", "lam1", "len1", "ang1")]),
    W.VBox([ctl[k] for k in ("w2", "s2", "mu2", "width2", "sharp2", "scale2")]),
])
for i, t in enumerate(["global", "diffuse", "fibrillar", "punctate"]):
    tabs.set_title(i, t)

display(W.HBox([tabs, out]))
W.interactive_output(cb, ctl)

### 1.2.2 Multiple Markers

In [ ]:
# Each marker is defined by this range of parameters
r = dict(
    amp=P(.1, 3, .1, 1.5, "amp"),
    polarity=P(-2, 2, .1, .4, "polarity"),
    pol_dir=P(-np.pi, np.pi, .1, .7, "pol dir"),

    w0=P(0, 1, .05, .50, "weight"),      s0=P(0, 2, .05, .4, "strength"),
    mu0=P(-1, 1.5, .05, 1.00, "mu"),     width0=P(.05, 1.5, .05, .25, "width"),
    sharp0=P(.5, 10, .5, 2.5, "sharp"),  scale0=P(.5, 12, .5, 4.0, "blob scale"),

    w1=P(0, 1, .05, .35, "weight"),      s1=P(0, 2, .05, 1.0, "strength"),
    mu1=P(-1, 1.5, .05, .45, "mu"),      width1=P(.05, 1.5, .05, .50, "width"),
    sharp1=P(.5, 10, .5, 2.0, "sharp"),  lam1=P(2, 25, .5, 8.0, "stripe lam"),
    len1=P(3, 40, 1., 25., "stripe len"), ang1=P(0, 180, 5., 30., "stripe angle"),

    w2=P(0, 1, .05, .15, "weight"),      s2=P(0, 2, .05, 1.1, "strength"),
    mu2=P(-1, 1.5, .05, -.50, "mu"),     width2=P(.05, 1.5, .05, .50, "width"),
    sharp2=P(.5, 10, .5, 3.0, "sharp"),  scale2=P(.5, 5, .1, 1.3, "fine scale")
)

g = dict(
    amp=P(.1, 3, .1, 1.3, "amp"),
    polarity=P(-2, 2, .1, -0.5, "polarity"),
    pol_dir=P(-np.pi, np.pi, .1, -1.2, "pol dir"),

    w0=P(0, 1, .05, .80, "weight"),      s0=P(0, 2, .05, .7, "strength"),
    mu0=P(-1, 1.5, .05, 0.50, "mu"),     width0=P(.05, 1.5, .05, .60, "width"),
    sharp0=P(.5, 10, .5, 1.5, "sharp"),  scale0=P(.5, 12, .5, 6.0, "blob scale"),

    w1=P(0, 1, .05, .65, "weight"),      s1=P(0, 2, .05, 0.5, "strength"),
    mu1=P(-1, 1.5, .05, .90, "mu"),      width1=P(.05, 1.5, .05, .30, "width"),
    sharp1=P(.5, 10, .5, 4.0, "sharp"),  lam1=P(2, 25, .5, 12.0, "stripe lam"),
    len1=P(3, 40, 1., 15., "stripe len"), ang1=P(0, 180, 5., 90., "stripe angle"),

    w2=P(0, 1, .05, .45, "weight"),      s2=P(0, 2, .05, 0.8, "strength"),
    mu2=P(-1, 1.5, .05, .20, "mu"),      width2=P(.05, 1.5, .05, .80, "width"),
    sharp2=P(.5, 10, .5, 5.0, "sharp"),  scale2=P(.5, 5, .1, 2.5, "fine scale")
)

b = dict(
    amp=P(.1, 3, .1, 1.2, "amp"),
    polarity=P(-2, 2, .1, 1.2, "polarity"),
    pol_dir=P(-np.pi, np.pi, .1, 0.0, "pol dir"),

    w0=P(0, 1, .05, .20, "weight"),      s0=P(0, 2, .05, 1.2, "strength"),
    mu0=P(-1, 1.5, .05, -0.50, "mu"),    width0=P(.05, 1.5, .05, .80, "width"),
    sharp0=P(.5, 10, .5, 5.5, "sharp"),  scale0=P(.5, 12, .5, 2.0, "blob scale"),

    w1=P(0, 1, .05, .85, "weight"),      s1=P(0, 2, .05, 1.5, "strength"),
    mu1=P(-1, 1.5, .05, -.90, "mu"),     width1=P(.05, 1.5, .05, 1.00, "width"),
    sharp1=P(.5, 10, .5, 1.0, "sharp"),  lam1=P(2, 25, .5, 1.0, "stripe lam"),
    len1=P(3, 40, 1., .35, "stripe len"), ang1=P(0, 180, 5., 145., "stripe angle"),

    w2=P(0, 1, .05, .75, "weight"),      s2=P(0, 2, .05, 1.8, "strength"),
    mu2=P(-1, 1.5, .05, .80, "mu"),      width2=P(.05, 1.5, .05, .20, "width"),
    sharp2=P(.5, 10, .5, 1.0, "sharp"),  scale2=P(.5, 5, .1, 4.3, "fine scale")
)

# from single_cell import single_cell
cell = generate_single_cell(size = 201, Tape = tape)

# To plot the marker, we build each channel and stack them along the depth axis
def plot_img(cell, p_r, p_g, p_b):
    cell, phi, tau = cell['cell'], cell['phi'], cell['tau']
    
    # Helper to unpack a parameter dict and render a single channel
    def render_channel(p_dict):
        # Build kwargs for the underlying components (excluding the top-level rendering params)
        comp_vals = {k: v.value for k, v in p_dict.items() if k not in ['amp', 'polarity', 'pol_dir']}
        comps = build_comps(**comp_vals)
        return render_marker(comps, cell, tau, phi, tape, p_dict['polarity'].value, p_dict['pol_dir'].value, p_dict['amp'].value)

    # Generate the R, G, B images
    img_r = render_channel(p_r)
    img_g = render_channel(p_g)
    img_b = render_channel(p_b)
    
    # Stack them into an (N, M, 3) RGB array
    rgb_composite = np.dstack((img_r, img_g, img_b))
    
    plt.imshow(render_cell(cell, [img_r, img_g, img_b]))
    plt.show()
    
plot_img(cell, r, g, b)

# 2) Synthetic tissue generation

In [ ]:
tissue_mask = support_mask(tape, scale_px=40.0, cover=0.55)
plt.imshow(tissue_mask)

In [ ]:
from scipy import ndimage as ndi

def build_tissue(tape, min_dist=11.0, radius=8.0, size_sigma=0.15, elong=1.6,
                 rough=0.15, beta=2.0, support_scale=40.0, cover=0.75, grow=1.35,
                 nuc_frac=0.35, nuc_frac_sigma=0.15, rim=1.5, nuc_corr=0.5, nuc_offset=0.9,):
    """Render a tile of packed cells: instance labels plus intrinsic coordinates.

    Nothing here reads pixel data. The output is a deterministic function of (tape, theta),
    which is what makes the label trustworthy as ground truth.

    Parameters
    ----------
    tape : dict
        Frozen randomness, drawn once and independent of every parameter below. Fields used
        here: xy (candidate centres), order (hard-core priority), a/b (cell harmonics),
        a2/b2 (independent nuclear harmonics), z_size, z_nucfrac, u_orient, u_offdir,
        u_offmag, support, tile, K. Parameters only THRESHOLD or SMOOTHLY MAP these, never
        resample them -- resampling per theta would make the fitting objective jagged.

    Point process
    -------------
    min_dist : float, px
        Hard-core radius: no two surviving centres are closer than this. GROUNDED -- measure
        it from real nearest-neighbour distances rather than fitting it, since it defines the
        mask. Roughly 1.2-1.8 x radius; too large and only a handful of candidates survive.
    support_scale : float, px
        Correlation length of the tissue-support field. Must stay >> cell size (>= ~4 x
        radius) so the support boundary can never be mistaken for a cell edge.
    cover : float in [0, 1]
        Fraction of the tile that is tissue. Applied as a quantile of the support field, so
        it maps monotonically onto realised coverage whatever the field's spread.

    Cell geometry (all mask-defining -> GROUNDED, not fitted)
    --------------------------------------------------------
    radius : float, px
        Median equivalent-circle radius of the FREE shape (area = pi r^2). Note the realised
        label area differs after packing -- ground this against post-packing median area,
        not against pi*radius^2.
    size_sigma : float
        Lognormal spread of cell size: realised r = radius * exp(size_sigma * z_size).
        0.15 gives roughly +-15%.
    elong : float >= 1
        Median aspect ratio. Area-preserving (the body-frame map has unit determinant), so
        this does not double as a size knob.
    rough : float
        Boundary irregularity, read as the SD of log radius: 0.15 ~ +-15% radial wobble.
        Above ~0.35 territories start pinching apart -- watch info["orphan_px"].
    beta : float
        Spectral tilt of the boundary harmonics at FIXED amplitude. Large -> a few fat
        lobes; small -> fine crenulation, which pinches at small radius. Below ~1.0 with
        radius < 5 px the shape is no longer band-limited for the pixel grid.

    Packing
    -------
    grow : float >= 1
        How far a cell may claim pixels, in units of its own free boundary (d = rho/r).
        1.0 -> free shapes with gaps between them; ~2 -> confluent, cells meeting along
        contact surfaces. In dense regions a neighbour binds first and grow is inert; in
        sparse regions grow alone sets the cell size.

    Nucleus
    -------
    nuc_frac : float
        Median nucleus:cell equivalent-radius ratio.
    nuc_frac_sigma : float
        Per-cell lognormal spread of that ratio. Set > 0, or nuclear area predicts cell area
        exactly and a nucleus-only model can invert the whole segmentation.
    nuc_corr : float in [0, 1]
        How much the nuclear outline mirrors the cell outline. 1 -> a scaled copy (leaks
        orientation and elongation); 0 -> independent. Implemented by mixing two frozen
        draws, so it stays smooth and safe to fit.
    nuc_offset : float in [0, 1]
        Nuclear eccentricity, as a fraction of the free cytoplasmic room. 0 puts the nucleus
        exactly on the tessellation seed, making the seed exactly recoverable from the
        nucleus.
    rim : float, px
        Minimum cytoplasmic clearance between the nuclear and plasma membranes. Enforced on
        the support function, so the labelled nuclear edge and the tau = 0 level set stay the
        same curve. Must stay resolvable -- below ~1 px it vanishes after the PSF.

    Returns
    -------
    labels : (tile, tile) int32       0 = background, k = cell k. THE GROUND TRUTH.
    nuc_labels : (tile, tile) int32   nuclei, same ids as `labels`.
    tau_img : (tile, tile) float      -1 nucleus centre, 0 nuclear envelope, +1 free
                                      membrane. Exceeds 1 in contact zones when grow > 1.
    phi_img : (tile, tile) float      body-frame angle, co-rotating with each cell.
    info : dict                       n_cells, centres, r_eff, support, orphan_px, empty,
                                      packing (cell pixels / support pixels).
    """
    tile = tape["tile"]
    # generate a support mask to limit the area where cells can be placed
    sup = support_mask(tape, support_scale, cover)
    # keep only the candidates that are sufficiently far apart and within the support mask
    keep = thin(tape, min_dist)
    cy0, cx0 = tape["xy"][keep].T
    keep = keep[sup[np.clip(cy0.astype(int), 0, tile-1),
                    np.clip(cx0.astype(int), 0, tile-1)]]

    best       = np.full((tile, tile), np.inf)
    labels     = np.zeros((tile, tile), np.int32)
    nuc_labels = np.zeros((tile, tile), np.int32)
    tau_img    = np.zeros((tile, tile))
    phi_img    = np.zeros((tile, tile))
    r_eff = radius * np.exp(size_sigma * tape["z_size"][keep])


    nf = np.clip(nuc_frac * np.exp(nuc_frac_sigma * tape["z_nucfrac"][keep]), 0.15, 0.85)

    for n, i in enumerate(keep, start=1):
        cy, cx = tape["xy"][i]
        
        f, (y0, x0) = stamp_cell(
            tape = tape, i = i, cy = cy, cx = cx, tile = tile, radius = r_eff[n-1], 
            nuc_frac = nf[n-1], rough = rough, elong = elong, 
            angle_deg = 180.0 * tape["u_orient"][i], beta = beta,
            grow=grow, rim=rim, nuc_corr=nuc_corr, nuc_offset=nuc_offset, 
        )
        h, w = f["d"].shape
        if h == 0 or w == 0:
            continue
        sl = (slice(y0, y0+h), slice(x0, x0+w))
        # tesselation to decide which cell is closest to each pixel 
        # -> find boundries

        win = (f["d"] < best[sl]) & (f["d"] <= grow) & sup[sl]
        best[sl] = np.where(win, f["d"], best[sl])
        labels[sl] = np.where(win, n, labels[sl])
        tau_img[sl] = np.where(win, f["tau"], tau_img[sl])
        phi_img[sl] = np.where(win, f["phi"], phi_img[sl])
        nuc_labels[sl] = np.where(win, np.where(f["nuc"], n, 0), nuc_labels[sl])

    # keep only the piece holding the seed; a two-piece "cell" is a wrong annotation
    orphan = 0
    for n, slc in enumerate(ndi.find_objects(labels), start=1):
        if slc is None:
            continue
        m = labels[slc] == n
        cc, k = ndi.label(m)
        if k > 1:
            main = 1 + int(np.argmax(np.bincount(cc.ravel())[1:]))
            drop = m & (cc != main)
            labels[slc][drop] = 0
            nuc_labels[slc][drop] = 0
            tau_img[slc][drop] = 0.0
            orphan += int(drop.sum())

    present = np.flatnonzero(np.bincount(labels.ravel(), minlength=len(keep)+1)[1:])
    info = dict(n_cells=len(keep), centres=tape["xy"][keep], r_eff=r_eff, support=sup,
                orphan_px=orphan, empty=len(keep)-len(present),
                packing=float((labels > 0).sum() / max(sup.sum(), 1)))
    return labels, nuc_labels, tau_img, phi_img, info



labels, nuc_labels, tau_img, phi_img, info = build_tissue(tape, min_dist=11.0, radius=8.0, size_sigma=0.15, elong=1.6,
                 rough=0.15, beta=1.2, support_scale=40.0, cover=0.95, grow=1.95)
plt.imshow(labels - nuc_labels)
plt.axis('off')
plt.show()
plot_tau(tau_img)
plot_polar_phi(phi_img, background_mask=(labels == 0))